# GPRO
不多说直接开撕

In [1]:
# 每日一导
import torch
import torch.nn.functional as F

In [2]:
def gather_log_probs(logits,labels):
    """
    logits: [batch, seq_len, vocab_size]
    labels: [batch, seq_len]
    return: [batch, seq_len]
    """
    log_probs = F.log_softmax(logits,dim=-1)
    selected_log_probs = log_probs.gather(
        dim= -1,
        index = labels.unsqueeze(-1)
    ).squeeze(-1)
    return selected_log_probs



In [4]:
def compute_group_advantages(
        rewards,
        group_size,
        eps = 1e-8,
):
    """
    rewards: [B * G]
    return: advantages [B * G]
    """
    assert rewards.ndim == 1
    assert rewards.size(0)%group_size == 0
    
    num_groups = rewards.size(0)//group_size
    rewards_grouped = rewards.view(num_groups,group_size)# [B,G]

    group_mean = rewards_grouped.mean(dim=1,keepdim=True)
    group_std = rewards_grouped.std(dim=1,keepdim=True,unbiased=False)
    advantages = (rewards_grouped - group_mean)/(group_std +eps)



In [5]:
def masked_mean(values,mask,dim=None,eps=1e-8):
    """
    values: [B, T]
    mask:   [B, T], 0/1
    """
    mask = mask.float()
    if dim is None:
        return (values*mask).sum()/(mask.sum()+eps)
    return (values*mask).sum(dim=dim)/(mask.sum(dim=dim)+eps)


In [6]:
def grpo_loss(
        old_log_probs,
        new_log_probs,
        advantages,
        response_mask,
        clip_eps=0.2,

):
    """
    old_log_probs: [B*G, T]
    new_log_probs: [B*G, T]
    advantages:    [B*G]
    response_mask: [B*G, T]
    """

    # PPO ratio
    log_ratio = new_log_probs -old_log_probs
    ratio = torch.exp(log_ratio)

    # seq level advantages -> broadcast to token
    adv = advantages.unsqueeze(1)

    # unclipped/clipped obj
    obj1 = ratio *adv
    obj2 = torch.clamp(ratio,1-clip_eps,1+clip_eps) *adv
    
    # max最大目标就是loss负号求最小
    token_loss = -torch.min(obj1,obj2)

    loss_per_seq = masked_mean(token_loss,response_mask,dim=1)
    loss = loss_per_seq.mean()

    return loss
    